# Week 1, Day 2: Introducing UnderCover

## Introduction

This notebook explores UnderCover Travel Insurance's claims data and compares language-model approaches to processing claims. It starts with data loading and exploratory analysis, evaluates direct structured LLM calls and optional model experiments, then builds a tool-using LangChain agent.

Use this notebook as a reference: each section explains the purpose of its code and how to interpret the results. Handler decisions are evaluation labels only; prompts sent to models contain the claim, policy record, and policy wording, not the handler's answer.

## Learning goals

- Load and inspect claims, policies, and evaluation labels.
- Build structured prompts and parse responses with a Pydantic schema.
- Compare model decisions against handler labels while tracking cost and latency.
- Define tools and run an agent that looks up a policy and records a decision.
- Recognize the limits of these experiments before using them to guide product decisions.

## Before you begin

Run this notebook from its saved location so the relative path to the data directory resolves. Install the project dependencies and set OPENROUTER_API_KEY in the repository's .env file before making model calls. LLM cells make network requests that may incur costs; check the models and expected spend before running the full experiments. Existing outputs may be stale, so rerun cells from top to bottom for results that match the current code.

## Notebook map

1. **Connect to OpenRouter** — load credentials and make a small API call.
2. **Load and examine the data** — read the queue, join labels, and inspect claim patterns.
3. **Run LLM experiments** — request structured decisions and compare accuracy, cost, and latency.
4. **Optional extras** — try other models or decision formats.
5. **Use a LangChain agent** — expose policy lookup and decision recording as tools, then compare it with direct calls.
6. **Conclusion** — summarize what the experiment demonstrates and what it does not.

In [1]:
# Standard-library helpers for project data, environment variables, timing, and decision types.
import json
import os
import time
from pathlib import Path
from typing import Literal

# Tabular analysis and interactive visualizations.
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
# LangChain agent, tool, and chat-model interfaces used in Part 4.
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field


## Connect to OpenRouter

Load the API credential and initialize the OpenAI-compatible OpenRouter client. The next small request is a connectivity smoke test; the remaining experiments make additional external requests that may incur cost.

In [2]:
# Load repository settings, allowing .env to override an existing shell value.
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if openrouter_api_key:
    print('OpenRouter API key loaded successfully.')
else:
    raise RuntimeError('OPENROUTER_API_KEY is missing; add it to the repository .env file.')

# OpenRouter exposes an OpenAI-compatible API endpoint.
openrouter = OpenAI(api_key=openrouter_api_key,
                        base_url=OPENROUTER_BASE_URL,
                        timeout=120,
                        max_retries=5)

OpenRouter API key loaded successfully.


In [3]:
# Keep system instructions separate from the user request.
messages = [
    {
        "role":"system",
        "content":"You are a helpful assistant who answers in the style of a pirate"
    },
    {
        "role": "user",
        "content": "Ahoy, matey! Tell me a pirate joke."
    }
]

# A small smoke test verifies credentials, connectivity, and the selected model.
response = openrouter.chat.completions.create(
            model="openai/gpt-6-luna",
            messages=messages
        )
display(Markdown(response.choices[0].message.content))


Why did the pirate buy an eye patch? Arrr, he couldn’t afford a whole new face! 🏴‍☠️

## Part 1: Load and examine the data

Load the claims queue, policy records, policy wording, and handler labels. Labels are retained for local comparison only and are not included in prompts sent to models.

In [4]:
from pathlib import Path
# Confirm the working directory before relying on the relative data path.
print(Path.cwd())

/Users/michaelperrine/Udemy/Donner/undercover/MyWork/UndercoverProject/week1


In [5]:
# This notebook lives three directories below the repository root.
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files"""
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False, held_out=False):
    """ Return a list of claims, and the handlers' decisions on them keyed by claim id.
    
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    # Handler labels are for scoring after inference and never enter model prompts.
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [7]:
# Set TINY=True to use the small subset for a quicker, lower-cost run.
TINY = False
sol = None

claims, labels = get_sample_data(tiny=TINY)
policies = {policy['policy_id']: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding='utf-8')

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [8]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

In [9]:
labels[claims[0]['claim_id']]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

### Explore the data

Create a table for comparison, inspect example claims, and plot how handler outcomes vary by claim type and difficulty. Each example should use a claim and its own matching label.

In [10]:
# Join each claim to its handler label to make one row per claim.
rows = []
for claim in claims:
    label = labels[claim['claim_id']]
    rows.append({
        'claim_id': claim['claim_id'],
        'summary': claim['summary'],
        'claim_type': claim['claim_type'],
        'amount_claimed': claim['amount_claimed'],
        'difficulty': label['difficulty'],
        'decision': label['decision'],
        'payout': label['payout']
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


In [11]:
home_alone = [claim for claim in claims if claim['claim_id'] == 'CLM-1012'][0]
# Look up the label for this same claim, not a different claim ID.
decision = labels[home_alone['claim_id']]

display(Markdown(f"""**{home_alone['summary']} 
{home_alone['story']} **Handler:** {decision['decision']}
${decision['payout']:,.2F}. {decision['reason']}"""))

**Valise stolen from a train at the Gare de Lyon 
On July 2 I took the train from Paris to Lausanne to join my husband, who was working there. At the Gare de Lyon I put my valise on the luggage rack in our carriage and stepped off the train to buy a bottle of water for the journey. I was gone for perhaps ten minutes. When I came back the valise was gone. It held my clothes and, far worse, all of my husband's writing: the printed manuscript of his novel, his notebooks and the backup drive, which I had packed so that he could work on everything in Switzerland. There are no other copies. I reported it to the police at the station that afternoon. I have valued the manuscript and notebooks at $1,900, though my husband would call that figure an insult. **Handler:** decline
$0.00. The valise was left on a luggage rack while the customer was off the train, out of her sight and reach in a place the public could get to. Belongings left unattended are not covered. The value placed on the manuscript would not have been payable in any case, since we pay for the physical item and not the work it holds.

In [13]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim['claim_id'] == 'CLM-1010'][0]
decision = queue_labels['CLM-1010']

display(Markdown(f"""**{hangover['summary']}**
{hangover['story']} **Handler:** {decision['decision']}
${decision['payout']:,.2f}. {decision['reason']} """))

**Emergency dental treatment in Las Vegas for a missing front tooth**
I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger. **Handler:** decline
$0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays. 

In [14]:
# Compare handler decisions across claim categories.
fig = px.histogram(data, 
                   x='claim_type',
                   color='decision',
                   title='Claims by type')
fig.show()

In [15]:
# Fix category order so difficulty proceeds from routine through manual.
fig = px.histogram(data,
                   x='difficulty',
                   color='decision',
                   category_orders={'difficulty': ['routine', 'hard', 'manual']},
                   title='Claims by difficulty')
fig.show()

### Part 1 summary

You loaded the claims queue and handler labels, inspected representative records, and plotted outcomes by claim type and difficulty. These charts describe this dataset; they do not establish model accuracy on unseen claims. Keep labels in the local scoring step rather than in prompts.

## Part 2: Run direct LLM experiments

Define a structured output schema, build a consistent prompt, and call models directly. Compare each run with handler labels, then summarize decision quality, cost, and elapsed time. Repeated runs can make many external API requests.

In [15]:
# Structured output makes downstream scoring predictable.
class ClaimDecision(BaseModel):
    """ A claims handler's decision on one claim."""

    rationale: str = Field(description='2-3 sentences to explain the decision along with the relevant section of the policy')
    decision: Literal['approve', 'decline', 'refer']
    payout: float = Field(description='The amount to pay  in US dollars, or 0 for a decline or a referral')

### Define the decision schema and prompt

The Pydantic schema constrains responses to the decision, payout, and rationale needed for comparison. The system prompt supplies the task and policy wording; each user prompt supplies only the selected claim and policy record.

In [16]:
# The system prompt holds task rules; policy wording is appended below.
INSTRUCTIONS = """ 
You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS} {policy_wording}"""

# Serialize only this claim and its policy record as user-specific context.
def make_user_prompt(claim):
    """ The customer's policy record ant the claim, as JSON."""
    policy = policies[claim['policy_id']]
    return f"""# Policy record

{json.dumps(policy, indent=2)} 

# Claim

{json.dumps(claim, indent=2)}
 """

def make_messages(claim):
    """ Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return[
        {'role': 'system',
         'content': SYSTEM_PROMPT
         },
         {
             'role': 'user',
             'content': make_user_prompt(claim)
         }
    ]

### Call one model

Measure one request, parse its response into the schema, and return a consistent record for later analysis.

In [17]:
def decide(claim, model, effort):
    """Request and normalize one structured claim decision."""

    # Ask one model to decide one claim.
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={'provider': {'require_parameters': True}},
    )
    # The response is parsed against ClaimDecision by the API client.
    answer = response.choices[0].message.parsed
    return {
        'model': model,
        'effort': effort,
        'technique': 'LLM',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 1)

    }

In [18]:
first = decide(claims[0], 'openai/gpt-6-luna', 'low')
first

{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'The cancellation due to a crew problem qualifies under Section 3, and the delay from the scheduled June 18 departure until the actual June 19 departure exceeds the Premium plan’s 6-hour waiting period. The attached carrier notice and receipts support the claimed reasonable accommodation, meal, and transport costs; the full $420 is within the $800 Trip Delay maximum, with no deductible under this benefit.',
 'cost': 0.0003953,
 'seconds': 2.3}

### Run multiple models

Run the same claim set through each model and reasoning effort. API usage scales with the number of models, settings, and claims, so review the queue size before running every cell.

In [19]:
def is_right(answer, label):
    """Require a matching decision and a payout within one US dollar."""
    return answer['decision'] == label['decision'] and abs(answer['payout'] - label['payout']) <= 1

def run(model, effort, claims, labels):
    """Evaluate one model configuration over claims and summarize resource use."""
    # Run the model on the claims and compare against the labels.
    results = []
    for claim in claims:
        answer = decide(claim, model,effort)
        label = labels[claim['claim_id']]
        results.append({**answer, 
                        'difficulty': label['difficulty'],
                        'right': is_right(answer, label)})
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [20]:
luna_low = run('openai/gpt-6-luna', 'low',claims, labels)

openai/gpt-6-luna at low: 20 of 25 right for $0.0046 in 78 seconds


In [21]:
luna_medium = run('openai/gpt-6-luna', 'medium', claims, labels)

openai/gpt-6-luna at medium: 19 of 25 right for $0.0065 in 123 seconds


In [22]:
luna_high = run('openai/gpt-6-luna', 'high', claims, labels)

openai/gpt-6-luna at high: 16 of 25 right for $0.0075 in 166 seconds


In [23]:
deepseek = run('deepseek/deepseek-v4.1-flash','high', claims, labels)

deepseek/deepseek-v4.1-flash at high: 20 of 25 right for $0.0354 in 288 seconds


In [24]:
gemini = run('google/gemini-3.5-flash-lite', 'low', claims, labels)

google/gemini-3.5-flash-lite at low: 15 of 25 right for $0.0334 in 31 seconds


In [25]:
# Spot-check Qwen with the same high reasoning effort as the batch run below.
start = time.time()

response = openrouter.with_options(
    timeout=60.0,
    max_retries=0,
).chat.completions.parse(
    model="qwen/qwen3.5-397b-a17b",
    messages=make_messages(claims[0]),
    response_format=ClaimDecision,
    extra_body={"reasoning": {"effort": "high"}},
    max_tokens=4096,
)

print(f"Completed in {time.time() - start:.1f} seconds")
print(response.choices[0].message.parsed)

Completed in 39.4 seconds
rationale='The claim is for Trip Delay under a Premium plan. The waiting period for Premium is 6 hours (Section 1). The flight was scheduled for 11:40 on June 18 and departed at 06:55 on June 19, a delay of approximately 19 hours, which exceeds the waiting period. The cause was a crew problem, which is a covered reason (Section 3). The expenses claimed (accommodation, meals, local transport) are covered reasonable extra costs (Section 3). All required documents (carrier confirmation and receipts) are attached (Section 3 & 11). The deductible does not apply to Trip Delay benefits (Section 1). The total claimed amount ($420) is within the Premium plan maximum for Trip Delay ($800).' decision='approve' payout=420.0


In [35]:
qwen = run('qwen/qwen3.5-397b-a17b', 'high', claims, labels)

qwen/qwen3.5-397b-a17b at high: 18 of 25 right for $0.3582 in 487 seconds


# Results by difficulty

Count correct results by model, reasoning effort, and difficulty. A result is correct only when both the decision matches and the payout is within one US dollar of the handler's amount.

In [36]:
# Combine model runs so each model/effort group can be compared.
scored = pd.DataFrame(luna_low + luna_medium + luna_high + deepseek + gemini + qwen )
scored.pivot_table(index=['model', 'effort'], 
                   columns='difficulty',
                   values='right',
                   aggfunc='sum',
                   sort=False)

difficulty                           routine  hard  manual
model                        effort                       
openai/gpt-6-luna            low          13     5       2
                             medium       14     5       0
                             high         13     3       0
deepseek/deepseek-v4.1-flash high         15     5       0
google/gemini-3.5-flash-lite low          14     1       0
qwen/qwen3.5-397b-a17b       high         15     3       0

### Inspect incorrect results

Show each model decision and rationale beside the handler label, reason, and evaluation checks. This makes failures reviewable instead of relying on a single accuracy count.

In [37]:
def show_misses(results, labels):
    """ Show each claim in a set of results that the model got wrong,
    with its rationale and the handler's reason.
    """
    # Keep claim-level explanations beside the corresponding handler assessment.
    for result in results:
        if not result['right']:
            label = labels[result['claim_id']]
            display(Markdown(f"""**{result['claim_id']},
            {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f},
            where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))

show_misses(luna_low, labels)


    

**CLM-1002,
            routine**: refer $0.00,
            where the handler said approve $580.00

- Model: The loss occurred during the covered trip, and the carrier report and settlement letter document the loss and the $400 carrier payment. However, the claim includes the toiletry kit and shirts without receipts, and section 11 requires the documents named for the benefit; the claim therefore needs to wait for the missing supporting documents. Under section 5, eligible items are subject to Premium’s $300 per-item limit, and section 10 deducts the carrier payment before applying any benefit maximum and deductible.
- Handler: The loss was reported to the carrier and the carrier has declared the bag lost. The suit is over the $300 per-item limit on Premium, so it counts at $300. The carrier's $400 comes off, then the $50 deductible.
- Tests: Payout arithmetic: the per-item limit, the carrier's payment and the deductible in order


**CLM-1004,
            routine**: refer $0.00,
            where the handler said approve $186.00

- Model: The carrier report confirms the checked baggage arrived 31 hours and 15 minutes after the traveler reached Edinburgh, exceeding the Premium plan’s 12-hour waiting period, and the required carrier report and receipts are attached. Section 4 covers essential clothing and toiletries purchased while waiting; the dress and underwear exceed what is clearly essential, so I would refer that portion for review, while the toiletries are eligible under the policy. Since the claim cannot be fully settled without reviewing the clothing expense, refer the claim.
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1019,
            manual**: approve $210.00,
            where the handler said refer $0.00

- Model: The carrier confirms a 9-hour delay caused by a crew problem, meeting the Premium plan’s 6-hour waiting period under Section 3. Receipts and carrier confirmation are attached; the $210 in reasonable accommodation, meal and local transport costs is below the $800 Trip Delay maximum, and no deductible applies.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020,
            manual**: refer $0.00,
            where the handler said approve $240.00

- Model: The claim concerns loss of personal belongings during the covered trip, but the account describes the sunglasses as stolen. Section 5 requires theft to be reported to local police within 24 hours and a copy of the police report; the attached tour-guide letter does not satisfy that requirement, so the claim needs review.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1022,
            manual**: refer $0.00,
            where the handler said approve $155.00

- Model: The carrier confirms a severe-weather delay of more than 30 hours during the covered trip, exceeding the First plan’s 3-hour waiting period, so the delay qualifies under Section 3. The $85 rebooking fee is supported by a receipt and is payable; the $140 meal expense lacks the required receipts, so that portion must be referred for the missing documentation under Sections 3 and 11.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


# Cost and latency

Aggregate correct results, API cost, and elapsed time for each model and reasoning effort. These sums describe this fixed test set; compare per-claim averages across runs with different claim counts.

In [38]:
# Sum correctness, cost, and elapsed time within each configuration.
scored.groupby(['model', 'effort'], sort=False)[['right', 
                                                 'cost', 
                                                 'seconds']].sum()

right      cost  seconds
model                        effort                          
openai/gpt-6-luna            low        20  0.004626     78.5
                             medium     19  0.006502    122.8
                             high       16  0.007500    166.2
deepseek/deepseek-v4.1-flash high       20  0.035423    288.5
google/gemini-3.5-flash-lite low        15  0.033361     31.3
qwen/qwen3.5-397b-a17b       high       18  0.358179    486.9

### Part 2 summary

The direct-call baseline returns a structured decision, payout, rationale, configuration, cost, and elapsed time for each claim. Review aggregate counts and individual misses: a matching decision with an incorrect payout is still scored as incorrect. Rankings can vary with the claims, wording, model version, prompt, and reasoning effort.

## Part 3: Optional extras

These experiments are not required for the core comparison. They add another model and a decision-format experiment; the latter evaluates decision choice only, not payout correctness.

### Optional: Sol at high reasoning effort

Run the standard structured-decision evaluation with this additional model. If skipped, the final comparison omits it automatically.

In [39]:
# This optional run is included in the final comparison if executed.
sol = run('openai/gpt-6-sol', 'high', claims, labels)

openai/gpt-6-sol at high: 18 of 25 right for $0.1239 in 121 seconds


### Optional: Jev decision format

Ask Jev for a constrained decision choice and confidence, then compare its choice to the handler label. This is not directly equivalent to earlier scores: Jev does not produce a payout or rationale here.

In [40]:
JEV = 'typesafe/jev-1.13'
JEV_QUESTION = {
    'type': 'choice',
    'instructions': "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    'criteria': {
        'approve': 'The wording covers the claim and the required documents are attached.',
        'decline': 'The wording does not cover the claim .',
        'refer': 'A colleague needs to look at it, because a required document is missing or the story and the documents disagree, or the wording does not settle the question.',

    },
}
openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)

def decide_with_jev(claim):
    """Request a choice and confidence, retaining the handler choice for scoring."""
    """ Ask Jev for the decision on one claim, with its confidence """
    state = {'policy_wording': policy_wording, 
             'policy': policies[claim['policy_id']],
             'claim': claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(
        model=JEV,
        state=state,
        questions={'decision': JEV_QUESTION})
    answer = response.answers['decision']
    return{
        'claim_id': claim['claim_id'],
        'decision': answer.choice,
        'confidence': answer.confidence,
        'right': answer.choice == labels[claim['claim_id']]['decision'],
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 2)
    }

In [41]:
# Jev is scored on decision choice only; direct LLM scoring also checks payout.
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result['decision'] == labels[result['claim_id']]['decision'] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Jev: 17 of 25 decisions right for $0.0033 in 7 seconds
Luna at low: 20 of 25 decisions right


In [ ]:
# Check whether high-confidence answers are more often correct.
jev['confident'] = jev['confidence'] >= 0.9
jev.groupby('confident')['right'].agg(['sum', 'count'])

,sum,count
confident,,
False,12,20
True,4,5


### Part 3 summary

The optional Sol run uses the same structured evaluation as the core models. Jev instead returns a choice and confidence, so its decision-only score is not directly comparable with metrics that also validate payout. Treat confidence as an experimental signal, not a guarantee of correctness.

## Part 4: Use LangChain create_agent

First make a simple LangChain chat request, then create tools for policy lookup and recording decisions. The agent receives the claim and policy wording, retrieves the policy through its tool, and records its result in an in-memory dictionary for this notebook run. This is a demonstration, not persistent storage.

In [42]:
# Demonstrate LangChain's chat-model interface before creating the agent.
llm = ChatOpenAI(model='openrouter/free', 
                 base_url=OPENROUTER_BASE_URL,
                 api_key=openrouter_api_key)
messages =[
    {
        'role': 'system',
        'content': 'You are a helpful assistant who answers in the style of a pirate'
    },
    {
        'role': 'user',
        'content': 'What is the meaning of the universe?'
    },
]
response = llm.invoke(messages)
display(Markdown(response.content))

Arrr, sailor, science can tell us **how** the universe works, but not necessarily **why** it exists.

Some answers say it is a grand accident without a built-in purpose. Others see purpose in consciousness, love, curiosity, beauty, or the stories we tell. Perhaps the universe itself has no fixed meaning—and perhaps that is what makes meaning something **we create together**, through how we treat one another and the world around us.

So, mate, the universe may not have a single answer… but it can still be a place worth paying attention to. 🌌

In [44]:
# Notebook-local stand-in for persistence; it resets when the kernel restarts.
recorded = {}

# Give the agent a narrow interface for policy lookup and decision recording.
@tool
def get_policy(policy_id: str) -> str:
    """ Look up a customer's policy record: the plan, 
    the add-ons, the purchase date, the trip and the travelers.
    """
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f'There is no policy {policy_id}'

@tool
def record_decision(claim_id: str, decision: Literal['approve', 'decline', 'refer'], payout: float, rationale: str) -> str:
    """ Record the decision on a claim in the claims system.
    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {'decision': decision, 
                          'payout': payout,
                          'rationale': rationale
                          }
    return f"Recorded {decision} on {claim_id}"

In [45]:
# Configure the model and system instructions for the claims agent.
luna = ChatOpenAI(model='openai/gpt-6-luna', 
                  base_url=OPENROUTER_BASE_URL,
                  api_key=openrouter_api_key,
                  reasoning_effort='low',
                  timeout=120,
                  max_retries=5)
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""

def make_claim_prompt(claim):
    """ The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f""" # Claim
    {json.dumps(claim, indent=2)}
"""

agent = create_agent(model=luna,
                     tools=[get_policy,
                            record_decision],
                            system_prompt=AGENT_SYSTEM_PROMPT)

In [46]:
# Inspect one agent trace to see its tool calls and final response.
result = agent.invoke({
    'messages':[{
        'role': 'user',
        'content': make_claim_prompt(claims[0])
    }]
})

for message in result['messages'][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_n2O3UqPbb42AW8WK7Hq6L4Sb)
 Call ID: call_n2O3UqPbb42AW8WK7Hq6L4Sb
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_4bQyUBa7nfIuSMyo7YGJdP35)
 Call ID: call_4bQyUBa7nfIuSMyo7YGJdP35
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420
    rationale: The carrier confirmed the June 18 return flight was 

In [47]:
def decide_with_agent(claim):
    """Run the agent once and score its result against the handler label."""
    """ Run the agent on one claim and check what it recorded against the handler's decision. """
    start = time.time()
    result = agent.invoke(
        {
            'messages': [
                {
                    'role': 'user',
                    'content': make_claim_prompt(claim)
                }
            ]
        })
    answer = recorded.get(claim['claim_id'],
                          {'decision':'nothing recorded',
                           'payout': 0.0,
                           'rationale': ""})
    label = labels[claim['claim_id']]
    # Sum costs across AI turns, including any tool-use cycle.
    cost = sum(message.response_metadata['token_usage']['cost'] for message in result['messages'] if message.type == 'ai')
    return{
        'model': 'openai/gpt-6-luna',
        'effort': 'low',
        'technique': 'Agent',
        'claim_id': claim['claim_id'],
        **answer,
        'difficulty': label['difficulty'],
        'right': is_right(answer, label),
        'cost': cost,
        'seconds': round(time.time() - start, 1),
    }

# Reset in-memory results before evaluating the full claim set.
recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision.")

25 of 25 claims have a recorded decision.


In [48]:
# Compare the direct-call baseline and agent on correctness, cost, and time.
pd.DataFrame(luna_low + agent_results).groupby(['model',
                                                'effort',
                                                'technique'],
                                                sort=False)[['right',
                                                             'cost',
                                                             'seconds']].sum()

right      cost  seconds
model             effort technique                          
openai/gpt-6-luna low    LLM           20  0.004626     78.5
                         Agent         17  0.008627    166.0

In [49]:
show_misses(agent_results, labels)

**CLM-1004,
            routine**: approve $74.00,
            where the handler said approve $186.00

- Model: Under section 4, Premium covers essential clothing and toiletries purchased while checked baggage is delayed at least 12 hours; the carrier report confirms a delay of over 31 hours, and the required receipts and report are attached. The $47 toiletries and $27 underwear are eligible, but the $112 dress bought for a welcome dinner is not essential clothing, so the payable amount is $74 and is within the $300 maximum; no deductible applies.
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1010,
            routine**: refer $0.00,
            where the handler said decline $0.00

- Model: The invoice and clinical notes are attached, but the notes do not establish exactly when or how the tooth was extracted; the claimant has no memory of the event, and the evidence of heavy drinking and impairment raises a question under the section 9 exclusion for losses while impaired. The notes also advise follow-up with the dentist at home, but do not establish whether the temporary bridge treatment could reasonably wait, as addressed by section 7; a colleague should resolve these coverage questions.
- Handler: The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.
- Tests: A general exclusion: a loss while impaired by alcohol, found in the clinical notes


**CLM-1018,
            manual**: approve $330.00,
            where the handler said refer $0.00

- Model: The theft occurred during the covered trip and was reported to local police within 24 hours; the attached Carabinieri report supports the reported loss, as required by section 5. The eligible amounts are $300 for the jacket (Premium per-item limit) and $80 for the daypack; after the $50 baggage deductible and with no other recovery reported, the payout is $330 under section 10.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019,
            manual**: approve $210.00,
            where the handler said refer $0.00

- Model: The Premium plan's 6-hour waiting period was met: the flight departed 9 hours 20 minutes late due to a crew duty-time problem, a covered crew cause under section 3, and the carrier confirmation and expense receipts are attached. The $210 in reasonable delay expenses is within the $800 Trip Delay maximum; section 10 requires no deductible for this benefit.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020,
            manual**: refer $0.00,
            where the handler said approve $240.00

- Model: The policy covers baggage loss during the covered trip, but theft must be reported to local police within 24 hours and a copy of the police report is required under Section 5. The attached guide's letter is not a police report, so the required document is missing; please review whether an acceptable alternative can be obtained before deciding payment.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021,
            hard**: approve $1,053.00,
            where the handler said approve $825.00

- Model: The theft occurred during the covered trip and was reported to local police the same day, with a police report attached as required by section 5. The eligible items total $1,103: Electronics Plus raises the electronics per-item limit to $1,200, and the tote is within the Premium $300 per-item limit; after the $50 Premium baggage deductible under sections 1 and 10, the payout is $1,053.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022,
            manual**: refer $0.00,
            where the handler said approve $155.00

- Model: The carrier confirms a severe-weather delay of over 30 hours, which exceeds the First plan’s 3-hour waiting period, and the $85 rebooking receipt is attached. Section 3 requires receipts for claimed extra costs; receipts are missing for the $140 meals, so the claim needs follow-up before payment can be determined.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1024,
            manual**: approve $250.00,
            where the handler said refer $0.00

- Model: The phone was stolen during the covered trip and the police report documents the theft on the same day, satisfying section 5; the claim was also filed within 90 days under section 11. Under sections 5 and 10, the Premium per-item limit reduces the $899 claim to $300, and the $50 deductible leaves a payout of $250; Electronics Plus was not purchased.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


### Put the experiments together

Compare the direct-call results with the agent using the same decision-and-payout correctness check. Optional experiments are included when they have been run.

In [50]:
# Build the main comparison from completed experiments.
experiments = [luna_low,
                luna_medium,
                gemini,
                deepseek,
                qwen,
                agent_results]

# Include the optional Sol run only if its cell has been executed.
if sol is not None:
    experiments.insert(3, sol)
rows = []
for results in experiments:
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    rows.append({
        'LLM': results[0]['model'],
        'reasoning': results[0]['effort'],
        'technique': results[0]['technique'],
        'accuracy': f"{right} / {len(results)}",
        'cost': f"${cost:.4f}",
        'spend': f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,reasoning,technique,accuracy,cost,spend
0,openai/gpt-6-luna,low,LLM,20 / 25,$0.0046,3.1 seconds a claim
1,openai/gpt-6-luna,medium,LLM,19 / 25,$0.0065,4.9 seconds a claim
2,google/gemini-3.5-flash-lite,low,LLM,15 / 25,$0.0334,1.3 seconds a claim
3,openai/gpt-6-sol,high,LLM,18 / 25,$0.1239,4.9 seconds a claim
4,deepseek/deepseek-v4.1-flash,high,LLM,20 / 25,$0.0354,11.5 seconds a claim
5,qwen/qwen3.5-397b-a17b,high,LLM,18 / 25,$0.3582,19.5 seconds a claim
6,openai/gpt-6-luna,low,Agent,17 / 25,$0.0086,6.6 seconds a claim


## Conclusion

This notebook builds a small, repeatable comparison between direct LLM calls and a tool-using agent on UnderCover's labeled claims. It demonstrates how structured outputs, consistent prompts, explicit scoring, and cost/latency tracking make experiments easier to inspect and compare.

The results are limited to the selected claims, labels, wording, model versions, prompts, and settings used here. They do not prove production readiness or parity with handlers. Before relying on an approach, inspect failures, rerun from a clean kernel, verify outputs and costs, and design broader evaluation that keeps handler labels out of model context. The agent's recorded dictionary is temporary and is not the product's claims system.